# Fintech Data Pipeline
**Student:** Paras Dwivedi | **ID:** bitsom_ftai_2601265

This notebook covers:
- Part 3: Python Reconciliation Workflow
- Part 4: JSON Normalization
- Dashboard Data Exports

## Part 3: Python Reconciliation Workflow

### Step 1 — Load Both Files

In [52]:
import pandas as pd
import json

# Load ledger and gateway files
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

print('Ledger shape:', ledger.shape)
print('Gateway shape:', gateway.shape)
print('\nLedger preview:')
print(ledger.head())
print('\nGateway preview:')
print(gateway.head())

Ledger shape: (10, 6)
Gateway shape: (9, 6)

Ledger preview:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       850.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R004       2026-03-02        M003      2100.0  success   
4           R005       2026-03-03        M004      7200.0  success   

  payment_method  
0            UPI  
1           Card  
2         Wallet  
3           Card  
4           Card  

Gateway preview:
  transaction_id transaction_date merchant_id  amount_usd   status  \
0           R001       2026-03-01        M001      1200.0  success   
1           R002       2026-03-01        M002       900.0  success   
2           R003       2026-03-02        M001       500.0  success   
3           R005       2026-03-03        M004      7200.0   failed   
4           R006       2026-03-03   

### Step 2 — Check Duplicates and Nulls

In [53]:
print('=== LEDGER ===')
print('Duplicates:', ledger.duplicated().sum())
print('Nulls:')
print(ledger.isnull().sum())

print('\n=== GATEWAY ===')
print('Duplicates:', gateway.duplicated().sum())
print('Nulls:')
print(gateway.isnull().sum())

=== LEDGER ===
Duplicates: 0
Nulls:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

=== GATEWAY ===
Duplicates: 0
Nulls:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


### Step 3 — Records Missing in Gateway

In [54]:
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])]
print('Missing in gateway:', len(missing_in_gateway))
print(missing_in_gateway)
missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)

Missing in gateway: 2
  transaction_id transaction_date merchant_id  amount_usd   status  \
3           R004       2026-03-02        M003      2100.0  success   
9           R010       2026-03-05        M004      2500.0  success   

  payment_method  
3           Card  
9         Wallet  


### Step 4 — Records Missing in Ledger

In [55]:
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])]
print('Missing in ledger:', len(missing_in_ledger))
print(missing_in_ledger)
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)

Missing in ledger: 1
  transaction_id transaction_date merchant_id  amount_usd   status  \
8           R011       2026-03-05        M003      1800.0  success   

  payment_method  
8           Card  


### Step 5 — Amount Mismatches

In [56]:
merged = ledger.merge(gateway, on='transaction_id', suffixes=('_ledger', '_gateway'))
amount_mismatches = merged[merged['amount_usd_ledger'] != merged['amount_usd_gateway']]
print('Amount mismatches:', len(amount_mismatches))
print(amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']])
amount_mismatches.to_csv('amount_mismatches.csv', index=False)

Amount mismatches: 2
  transaction_id  amount_usd_ledger  amount_usd_gateway
1           R002              850.0               900.0
6           R008              640.0               600.0


### Step 6 — Status Mismatches

In [57]:
status_mismatches = merged[merged['status_ledger'] != merged['status_gateway']]
print('Status mismatches:', len(status_mismatches))
print(status_mismatches[['transaction_id', 'status_ledger', 'status_gateway']])
status_mismatches.to_csv('status_mismatches.csv', index=False)

Status mismatches: 1
  transaction_id status_ledger status_gateway
3           R005       success         failed


### Step 7 — Final Reconciliation Report

In [58]:
all_ids = pd.concat([ledger[['transaction_id']], gateway[['transaction_id']]]).drop_duplicates()

report_rows = []
for tid in all_ids['transaction_id']:
    l = ledger[ledger['transaction_id'] == tid]
    g = gateway[gateway['transaction_id'] == tid]

    l_amt = l['amount_usd'].values[0] if len(l) > 0 else None
    g_amt = g['amount_usd'].values[0] if len(g) > 0 else None
    l_status = l['status'].values[0] if len(l) > 0 else None
    g_status = g['status'].values[0] if len(g) > 0 else None

    if len(l) == 0:
        recon_status = 'missing_in_ledger'
    elif len(g) == 0:
        recon_status = 'missing_in_gateway'
    elif l_amt != g_amt:
        recon_status = 'amount_mismatch'
    elif l_status != g_status:
        recon_status = 'status_mismatch'
    else:
        recon_status = 'matched'

    report_rows.append({
        'transaction_id': tid,
        'ledger_amount': l_amt,
        'gateway_amount': g_amt,
        'ledger_status': l_status,
        'gateway_status': g_status,
        'reconciliation_status': recon_status
    })

recon_report = pd.DataFrame(report_rows)
print(recon_report)
recon_report.to_csv('reconciliation_report.csv', index=False)

   transaction_id  ledger_amount  gateway_amount ledger_status gateway_status  \
0            R001         1200.0          1200.0       success        success   
1            R002          850.0           900.0       success        success   
2            R003          500.0           500.0       success        success   
3            R004         2100.0             NaN       success           None   
4            R005         7200.0          7200.0       success         failed   
5            R006          950.0           950.0       success        success   
6            R007         3300.0          3300.0        failed         failed   
7            R008          640.0           600.0       success        success   
8            R009         4100.0          4100.0       success        success   
9            R010         2500.0             NaN       success           None   
10           R011            NaN          1800.0          None        success   

   reconciliation_status  


### Step 8 — Summary Metrics

In [59]:
issue_ids = recon_report[recon_report['reconciliation_status'] != 'matched']['transaction_id']
amount_at_risk = recon_report[recon_report['transaction_id'].isin(issue_ids)]['ledger_amount'].sum()

summary = {
    'total_ledger_rows': len(ledger),
    'total_gateway_rows': len(gateway),
    'missing_in_gateway_count': len(missing_in_gateway),
    'missing_in_ledger_count': len(missing_in_ledger),
    'amount_mismatch_count': len(amount_mismatches),
    'status_mismatch_count': len(status_mismatches),
    'reconciliation_issue_count': len(recon_report[recon_report['reconciliation_status'] != 'matched']),
    'ledger_total_amount': round(ledger['amount_usd'].sum(), 2),
    'gateway_total_amount': round(gateway['amount_usd'].sum(), 2),
    'amount_at_risk': round(float(amount_at_risk), 2)
}

print(json.dumps(summary, indent=2))
with open('summary_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\nsummary_metrics.json saved!')

{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 6,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 20550.0,
  "amount_at_risk": 13290.0
}

summary_metrics.json saved!


---
## Part 4: JSON Normalization

### Step 1 — Read JSON File

In [60]:
with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

print('Top-level keys:', list(api_data.keys()))
print('Number of batches:', len(api_data['batches']))

Top-level keys: ['generated_at', 'source', 'batches']
Number of batches: 2


### Step 2 — Flatten Nested JSON

In [61]:
rows = []
for batch in api_data['batches']:
    for settlement in batch['settlements']:
        rows.append({
            'batch_id': batch['batch_id'],
            'merchant_id': batch['merchant']['merchant_id'],
            'merchant_name': batch['merchant']['merchant_name'],
            'region': batch['merchant']['region'],
            'settlement_id': settlement['settlement_id'],
            'amount_usd': settlement['amount_usd'],
            'status': settlement['status'],
            'processed_at': settlement['processed_at'],
            'bank_name': settlement['bank']['name'],
            'bank_country': settlement['bank']['country']
        })

api_df = pd.DataFrame(rows)
print('Shape:', api_df.shape)
print(api_df)

Shape: (6, 10)
  batch_id merchant_id  merchant_name region settlement_id  amount_usd  \
0     B001        M001     Alpha Mart   APAC          S001      1520.5   
1     B001        M001     Alpha Mart   APAC          S002       980.0   
2     B001        M001     Alpha Mart   APAC          S003       640.0   
3     B002        M004  Delta Travels     US          S004      2100.0   
4     B002        M004  Delta Travels     US          S005       500.0   
5     B002        M004  Delta Travels     US          S006      7200.0   

    status          processed_at bank_name bank_country  
0  settled  2026-03-07T08:10:00Z    Bank A           IN  
1  pending  2026-03-07T08:45:00Z    Bank A           IN  
2  settled  2026-03-07T09:15:00Z    Bank B           SG  
3  settled  2026-03-07T08:20:00Z    Bank C           US  
4   failed  2026-03-07T08:50:00Z    Bank C           US  
5  settled  2026-03-07T09:30:00Z    Bank C           US  


### Step 3 — Clean Column Names & Convert Datetime

In [62]:
api_df.columns = [col.lower().replace(' ', '_') for col in api_df.columns]
api_df['processed_at'] = pd.to_datetime(api_df['processed_at'])
print(api_df.dtypes)
print(api_df)

batch_id                      object
merchant_id                   object
merchant_name                 object
region                        object
settlement_id                 object
amount_usd                   float64
status                        object
processed_at     datetime64[ns, UTC]
bank_name                     object
bank_country                  object
dtype: object
  batch_id merchant_id  merchant_name region settlement_id  amount_usd  \
0     B001        M001     Alpha Mart   APAC          S001      1520.5   
1     B001        M001     Alpha Mart   APAC          S002       980.0   
2     B001        M001     Alpha Mart   APAC          S003       640.0   
3     B002        M004  Delta Travels     US          S004      2100.0   
4     B002        M004  Delta Travels     US          S005       500.0   
5     B002        M004  Delta Travels     US          S006      7200.0   

    status              processed_at bank_name bank_country  
0  settled 2026-03-07 08:10:00+00:0

### Step 4 — Save api_normalized.csv

In [63]:
api_df.to_csv('api_normalized.csv', index=False)
print('api_normalized.csv saved! Shape:', api_df.shape)

api_normalized.csv saved! Shape: (6, 10)


---
## Dashboard Data Exports

### Load Cleaned Transactions

In [64]:
ct = pd.read_csv('cleaned_transactions.csv')
ct['transaction_date'] = pd.to_datetime(ct['transaction_date'])
captured = ct[ct['status'] == 'captured']
print('Total rows:', len(ct))
print('Captured rows:', len(captured))

Total rows: 30
Captured rows: 19


### Daily Summary

In [65]:
daily = ct.groupby('transaction_date').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count'),
    successful_count=('status', lambda x: (x == 'captured').sum())
).reset_index()
print(daily)
daily.to_csv('daily_summary.csv', index=False)

  transaction_date  total_gmv_usd  transaction_count  successful_count
0       2026-03-01        26382.0                  5                 5
1       2026-03-02        25049.0                  6                 3
2       2026-03-03        18391.0                  5                 4
3       2026-03-04        16420.0                  5                 4
4       2026-03-05        19232.0                  6                 1
5       2026-03-06        10606.0                  3                 2


### Payment Method Breakdown

In [66]:
pm = captured.groupby('payment_method').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()
print(pm)
pm.to_csv('payment_method_breakdown.csv', index=False)

  payment_method  total_gmv_usd  transaction_count
0           Card        34283.0                  8
1     NetBanking        11709.0                  2
2            UPI        28527.0                  6
3         Wallet         7836.5                  3


### Region Breakdown

In [67]:
rb = ct.groupby('default_region').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count'),
    avg_risk_score=('risk_score', 'mean')
).reset_index()
print(rb)
rb.to_csv('region_breakdown.csv', index=False)

  default_region  total_gmv_usd  transaction_count  avg_risk_score
0           APAC        82594.0                 22        65.47619
1             EU        18886.0                  4        47.25000
2             US        14600.0                  4        48.75000


### Merchant Performance Summary

In [68]:
mp = ct.groupby('merchant_name').agg(
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[ct.loc[x.index, 'status'] == 'captured'].sum()),
    chargeback_count=('status', lambda x: (x == 'chargeback').sum()),
    high_risk_count=('high_risk_flag', 'sum')
).reset_index()
print(mp)
mp.to_csv('merchant_performance_summary.csv', index=False)
print('\nAll dashboard CSVs saved!')

   merchant_name  total_gmv_usd  captured_gmv_usd  chargeback_count  \
0     Alpha Mart        40812.0           29984.5                 1   
1    Beta Stores        41782.0           33431.0                 1   
2    City Pharma         8640.0            8640.0                 0   
3  Delta Travels        14600.0           10300.0                 1   
4       Eco Home        10246.0               0.0                 1   

   high_risk_count  
0                2  
1                5  
2                0  
3                1  
4                1  

All dashboard CSVs saved!
